1. Определитесь, какую задачу будет решать ваша нейронная сеть.
2. Продумайте интерфейс взаимодействия с пользователем, какими параметрами модели пользователь будет управлять.
3. Обучите модель на любом публичном датасете или возьмите из любого предыдущего урока. Вспомните как происходит загрузка и выгрузка моделей в Keras.
4. Загрузите обученную модель в Colab с интерфейсом (деплой модели).
5. Создайте интерфейс для инференса вашей модели (для запросов к модели).
6. Изучите как происходит загрузка файлов для моделей с помощью Streamlit по [ссылке](https://docs.streamlit.io/develop/api-reference/widgets/st.file_uploader).
7. Добавьте в интерфейс возможность загрузки пользовательских данных для инференса. Это может быть текстовый файл, картинка, аудиофайл или др.
8. Выполнив задание, получите 3 балла.
9. Вы также можете получить дополнительные 2 балла, если реализуете в одном интерфейсе обучение модели и её инференс.

**Инференс** - это процесс исполнения обученных моделей машинного обучения для получения предсказаний на данных, поданных на вход модели.
Обычно нейронная сеть проходит три жизненных этапа: обучение, деплой и инференс. Инференсом называется непрерывная работа какой-либо нейронной сети на конечном устройстве.

**Деплой** - загрузка на сервер.

In [ ]:
pip install -U gradio tensorflow scikit-learn pandas matplotlib joblib

In [ ]:
# Импорты
import os
import random
from pathlib import Path

import gradio as gr
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Gradio:", gr.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


# ===
iris = load_iris()

FEATURE_NAMES = [
    "sepal_length",
    "sepal_width",
    "petal_length",
    "petal_width",
]

CLASS_NAMES = [str(name) for name in iris.target_names]

X = iris.data.astype(np.float32)
y = iris.target.astype(np.int32)

iris_df = pd.DataFrame(X, columns=FEATURE_NAMES)
iris_df["species"] = [CLASS_NAMES[i] for i in y]

print("Размер:", iris_df.shape)
print("Классы:", CLASS_NAMES)
display(iris_df.head())


# ===
MODEL_PATH = Path("/content/iris_gradio_model.keras")
SCALER_PATH = Path("/content/iris_gradio_scaler.joblib")
BATCH_RESULT_PATH = Path("/content/iris_batch_predictions.csv")
SAMPLE_CSV_PATH = Path("/content/iris_sample_for_inference.csv")

deployed_model = None
deployed_scaler = None


# ===
def build_model(
    learning_rate=0.01,
    hidden_units_1=32,
    hidden_units_2=16,
):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(4,), name="iris_features"),
        tf.keras.layers.Dense(int(hidden_units_1), activation="relu"),
        tf.keras.layers.Dense(int(hidden_units_2), activation="relu"),
        tf.keras.layers.Dense(3, activation="softmax", name="species"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=float(learning_rate)
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ===
def train_and_deploy(
    epochs=120,
    batch_size=16,
    learning_rate=0.01,
    hidden_units_1=32,
    hidden_units_2=16,
):
    global deployed_model, deployed_scaler

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=SEED,
        stratify=y,
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    ).astype(np.float32)

    X_test_scaled = scaler.transform(
        X_test
    ).astype(np.float32)

    new_model = build_model(
        learning_rate=learning_rate,
        hidden_units_1=hidden_units_1,
        hidden_units_2=hidden_units_2,
    )

    history = new_model.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=int(epochs),
        batch_size=int(batch_size),
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_accuracy",
                mode="max",
                patience=20,
                restore_best_weights=True,
                verbose=0,
            )
        ],
        verbose=0,
    )

    y_prob = new_model.predict(
        X_test_scaled,
        verbose=0,
    )

    y_pred = np.argmax(
        y_prob,
        axis=1,
    )

    accuracy = accuracy_score(
        y_test,
        y_pred,
    )

    report = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    report_df = (
        pd.DataFrame(report)
        .T
        .reset_index()
        .rename(columns={"index":"class"})
    )

    # Сохранение модели и scaler.
    new_model.save(MODEL_PATH)
    joblib.dump(scaler, SCALER_PATH)

    # DEPLOY: загружаем сохранённые файлы обратно.
    deployed_model = tf.keras.models.load_model(
        MODEL_PATH
    )
    deployed_scaler = joblib.load(
        SCALER_PATH
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12, 4),
    )

    axes[0].plot(
        history.history["loss"],
        label="train",
    )
    axes[0].plot(
        history.history["val_loss"],
        label="validation",
    )
    axes[0].set_title("Loss")
    axes[0].grid()
    axes[0].legend()

    axes[1].plot(
        history.history["accuracy"],
        label="train",
    )
    axes[1].plot(
        history.history["val_accuracy"],
        label="validation",
    )
    axes[1].set_title("Accuracy")
    axes[1].grid()
    axes[1].legend()

    plt.tight_layout()

    status = (
        "Модель обучена, сохранена и загружена для инференса.\n"
        f"TEST accuracy: {accuracy * 100:.2f}%\n"
        f"Эпох выполнено: {len(history.history['loss'])}\n"
        f"Модель: {MODEL_PATH.name}"
    )

    return (
        status,
        report_df,
        fig,
        str(MODEL_PATH),
    )


# ===
initial_status, initial_report, initial_plot, _ = train_and_deploy(
    epochs=120,
    batch_size=16,
    learning_rate=0.01,
    hidden_units_1=32,
    hidden_units_2=16,
)

print(initial_status)
display(initial_report)


# ===
def predict_single(
    sepal_length,
    sepal_width,
    petal_length,
    petal_width,
):
    if deployed_model is None or deployed_scaler is None:
        raise gr.Error("Модель ещё не загружена.")

    features = np.array([[
        float(sepal_length),
        float(sepal_width),
        float(petal_length),
        float(petal_width),
    ]], dtype=np.float32)

    scaled = deployed_scaler.transform(
        features
    ).astype(np.float32)

    probabilities = deployed_model.predict(
        scaled,
        verbose=0,
    )[0]

    class_id = int(
        np.argmax(probabilities)
    )

    probability_dict = {
        CLASS_NAMES[i]: float(p)
        for i, p in enumerate(probabilities)
    }

    return CLASS_NAMES[class_id], probability_dict


# ===
def predict_csv(file_path):
    if file_path is None:
        raise gr.Error("Сначала загрузите CSV-файл.")

    if deployed_model is None or deployed_scaler is None:
        raise gr.Error("Модель ещё не загружена.")

    df = pd.read_csv(file_path)

    missing = [
        column
        for column in FEATURE_NAMES
        if column not in df.columns
    ]

    if missing:
        raise gr.Error(
            "В CSV отсутствуют колонки: "
            + ", ".join(missing)
        )

    try:
        features = df[
            FEATURE_NAMES
        ].astype(np.float32)
    except ValueError as error:
        raise gr.Error(
            "Признаки должны быть числовыми."
        ) from error

    scaled = deployed_scaler.transform(
        features
    ).astype(np.float32)

    probabilities = deployed_model.predict(
        scaled,
        verbose=0,
    )

    class_ids = np.argmax(
        probabilities,
        axis=1,
    )

    result = df.copy()

    result["predicted_class"] = [
        CLASS_NAMES[i]
        for i in class_ids
    ]

    result["confidence"] = np.max(
        probabilities,
        axis=1,
    )

    result.to_csv(
        BATCH_RESULT_PATH,
        index=False,
    )

    return result, str(BATCH_RESULT_PATH)


# ===
sample_df = pd.DataFrame([
    {
        "sepal_length": 5.1,
        "sepal_width": 3.5,
        "petal_length": 1.4,
        "petal_width": 0.2,
    },
    {
        "sepal_length": 6.0,
        "sepal_width": 2.9,
        "petal_length": 4.5,
        "petal_width": 1.5,
    },
    {
        "sepal_length": 6.7,
        "sepal_width": 3.1,
        "petal_length": 5.6,
        "petal_width": 2.4,
    },
])

sample_df.to_csv(
    SAMPLE_CSV_PATH,
    index=False,
)

print("Пример:", SAMPLE_CSV_PATH)


# ===
with gr.Blocks(
    title="Iris Neural Network",
    analytics_enabled=False,
) as demo:

    gr.Markdown(
        '''
        # Нейронная сеть Iris

        Полный жизненный цикл:
        **обучение → сохранение → деплой → инференс**
        '''
    )

    # Вкладка обучения
    with gr.Tab("Обучение модели"):

        gr.Markdown(
            "Измените параметры и переобучите модель."
        )

        with gr.Row():
            epochs_input = gr.Slider(
                20,
                300,
                value=120,
                step=10,
                label="Количество эпох",
            )

            batch_input = gr.Dropdown(
                choices=[8, 16, 32, 64],
                value=16,
                label="Batch size",
            )

            lr_input = gr.Dropdown(
                choices=[
                    0.001,
                    0.003,
                    0.005,
                    0.01,
                ],
                value=0.01,
                label="Learning rate",
            )

        with gr.Row():
            units_1_input = gr.Slider(
                8,
                128,
                value=32,
                step=8,
                label="Нейронов Dense #1",
            )

            units_2_input = gr.Slider(
                4,
                64,
                value=16,
                step=4,
                label="Нейронов Dense #2",
            )

        train_button = gr.Button(
            "Обучить / переобучить модель",
            variant="primary",
        )

        training_status = gr.Textbox(
            label="Результат обучения",
            value=initial_status,
            lines=5,
        )

        training_report = gr.Dataframe(
            label="Classification report",
            value=initial_report,
            interactive=False,
        )

        training_plot = gr.Plot(
            label="Графики обучения",
            value=initial_plot,
        )

        model_download = gr.File(
            label="Скачать модель .keras",
            value=str(MODEL_PATH),
            interactive=False,
        )

        train_button.click(
            fn=train_and_deploy,
            inputs=[
                epochs_input,
                batch_input,
                lr_input,
                units_1_input,
                units_2_input,
            ],
            outputs=[
                training_status,
                training_report,
                training_plot,
                model_download,
            ],
        )

    # Вкладка инференса
    with gr.Tab("Инференс"):

        gr.Markdown("## Один цветок")

        with gr.Row():
            sepal_length_input = gr.Number(
                value=5.1,
                label="Sepal length",
            )
            sepal_width_input = gr.Number(
                value=3.5,
                label="Sepal width",
            )
            petal_length_input = gr.Number(
                value=1.4,
                label="Petal length",
            )
            petal_width_input = gr.Number(
                value=0.2,
                label="Petal width",
            )

        predict_button = gr.Button(
            "Определить вид",
            variant="primary",
        )

        single_class_output = gr.Textbox(
            label="Предсказанный класс",
        )

        single_prob_output = gr.Label(
            label="Вероятности",
            num_top_classes=3,
        )

        predict_button.click(
            fn=predict_single,
            inputs=[
                sepal_length_input,
                sepal_width_input,
                petal_length_input,
                petal_width_input,
            ],
            outputs=[
                single_class_output,
                single_prob_output,
            ],
        )

        gr.Markdown(
            '''
            ---
            ## Пакетный инференс из CSV

            Нужные колонки:
            `sepal_length, sepal_width, petal_length, petal_width`
            '''
        )

        sample_file = gr.File(
            label="Скачать пример CSV",
            value=str(SAMPLE_CSV_PATH),
            interactive=False,
        )

        csv_upload = gr.File(
            label="Загрузите свой CSV",
            file_types=[".csv"],
            type="filepath",
        )

        csv_predict_button = gr.Button(
            "Выполнить инференс CSV"
        )

        batch_table_output = gr.Dataframe(
            label="Результаты",
            interactive=False,
        )

        batch_file_output = gr.File(
            label="Скачать CSV с предсказаниями"
        )

        csv_predict_button.click(
            fn=predict_csv,
            inputs=[csv_upload],
            outputs=[
                batch_table_output,
                batch_file_output,
            ],
        )


# ===
demo.launch(
    share=True,
    debug=True,
)


TensorFlow: 2.20.0
Gradio: 6.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Размер: (150, 5)
Классы: ['setosa', 'versicolor', 'virginica']


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


Модель обучена, сохранена и загружена для инференса.
TEST accuracy: 93.33%
Эпох выполнено: 26
Модель: iris_gradio_model.keras


,class,precision,recall,f1-score,support
0,setosa,1.000000,1.000000,1.000000,10.000000
1,versicolor,0.900000,0.900000,0.900000,10.000000
2,virginica,0.900000,0.900000,0.900000,10.000000
3,accuracy,0.933333,0.933333,0.933333,0.933333
4,macro avg,0.933333,0.933333,0.933333,30.000000
5,weighted avg,0.933333,0.933333,0.933333,30.000000


Пример: /content/iris_sample_for_inference.csv
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c6a5a8280c6f275325.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c6a5a8280c6f275325.gradio.live
